# 03. Xây dựng Pipeline, Huấn luyện Mô hình & Tinh chỉnh Siêu tham số (Optuna)
**Đề tài:** Dự đoán khả năng khách hàng hủy đặt phòng khách sạn (*Hotel Booking Demand*)

## Kiến trúc thực nghiệm
1. **Tiền xử lý chống rò rỉ trong CV (`ColumnTransformer`)**:
   - Biến số: `SimpleImputer(strategy='median')` (+ `StandardScaler` cho `LogisticRegression`).
   - Biến phân loại ít giá trị (`low_card_cat_cols`): `OneHotEncoder(handle_unknown='ignore')`.
   - Biến phân loại nhiều giá trị (`country`, `agent`, `company`): `TargetEncoder(smooth='auto', cv=5)` khớp **nghiêm ngặt bên trong từng fold huấn luyện**.
2. **Bậc thang Mô hình (5-Fold Stratified CV trên `train_strat.csv`)**:
   - Baseline: `Logistic Regression`, `Decision Tree`.
   - Cây nâng cao & GBDT cơ sở: `Random Forest`, `XGBoost`, `LightGBM`, `CatBoost`.
3. **Tinh chỉnh siêu tham số bằng Optuna (`TPESampler` + `MedianPruner`)**: Tối ưu 5-Fold CV `ROC-AUC` cho Top 3 GBDT (`LightGBM`, `XGBoost`, `CatBoost`).
4. **Lưu trữ kết quả**: Chọn mô hình tốt nhất tổng thể, huấn luyện lại trên toàn bộ `train_strat.csv` và lưu mô hình (`.joblib`), dự đoán OOF (`.pkl`), bảng điểm CV (`.csv`) cùng tham số tối ưu (`.json`) vào `models/`.

In [1]:
from pathlib import Path
import json
import pickle
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import optuna

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with (PROCESSED_DIR / "feature_metadata.json").open("r", encoding="utf-8") as f:
    meta = json.load(f)

target_col = meta["target_col"]
num_cols = meta["num_cols"]
low_card_cols = meta["low_card_cat_cols"]
high_card_cols = meta["high_card_cat_cols"]
cat_dtypes = {c: str for c in low_card_cols + high_card_cols}

train_df = pd.read_csv(PROCESSED_DIR / "train_strat.csv", dtype=cat_dtypes)
X_train = train_df[num_cols + low_card_cols + high_card_cols].copy()
y_train = train_df[target_col].astype(int).values

print(f"Tập huấn luyện train_strat: {X_train.shape}, tỷ lệ hủy: {y_train.mean():.4f}")
print(f"Đặc trưng: {len(num_cols)} số | {len(low_card_cols)} low-card | {len(high_card_cols)} high-card")

Tập huấn luyện train_strat: (69576, 45), tỷ lệ hủy: 0.2731
Đặc trưng: 34 số | 8 low-card | 3 high-card


## 1. Hàm xây dựng Preprocessor & Chuẩn bị 5-Fold Stratified CV
Để đảm bảo không có rò rỉ dữ liệu nhưng vẫn đạt tốc độ cao khi chạy Optuna, ta chuẩn bị sẵn `build_preprocessor()` và tính trước các ma trận đã biến đổi theo đúng 5 fold độc lập (mỗi fold chỉ `fit` `TargetEncoder` và `OneHotEncoder` trên phần train của fold đó). Cuối cùng, mọi mô hình lưu xuống ổ đĩa đều được đóng gói thành `Pipeline` hoàn chỉnh fit trên toàn bộ `X_train`.

In [2]:
def build_preprocessor(scale_num: bool = False) -> ColumnTransformer:
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_num:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_steps), num_cols),
            ("low_cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), low_card_cols),
            ("high_cat", TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=RANDOM_STATE), high_card_cols),
        ],
        remainder="drop",
    )


def compute_metrics(y_true, y_proba, threshold: float = 0.50) -> dict:
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "log_loss": log_loss(y_true, np.clip(y_proba, 1e-7, 1 - 1e-7)),
    }


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv.split(X_train, y_train))

# Pre-fit fold-specific preprocessors strictly inside each fold (zero leakage)
fold_data_unscaled = []
fold_data_scaled = []
t0 = time.time()
for tr_idx, va_idx in cv_splits:
    X_tr_raw, X_va_raw = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    prep_u = build_preprocessor(scale_num=False)
    X_tr_u = prep_u.fit_transform(X_tr_raw, y_tr).astype(np.float32)
    X_va_u = prep_u.transform(X_va_raw).astype(np.float32)
    fold_data_unscaled.append((tr_idx, va_idx, X_tr_u, y_tr, X_va_u, y_va))

    prep_s = build_preprocessor(scale_num=True)
    X_tr_s = prep_s.fit_transform(X_tr_raw, y_tr).astype(np.float32)
    X_va_s = prep_s.transform(X_va_raw).astype(np.float32)
    fold_data_scaled.append((tr_idx, va_idx, X_tr_s, y_tr, X_va_s, y_va))

print(f"Hoàn tất tiền xử lý 5-Fold CV chống rò rỉ trong {time.time() - t0:.2f}s | Số chiều sau mã hóa: {fold_data_unscaled[0][2].shape[1]}")


def run_cv_model(model_name: str, estimator, use_scaled: bool = False):
    folds = fold_data_scaled if use_scaled else fold_data_unscaled
    oof_proba = np.zeros(len(y_train), dtype=np.float64)
    fold_metrics = []
    start = time.time()

    for _, va_idx, X_tr, y_tr, X_va, y_va in folds:
        clf = clone(estimator)
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_va)[:, 1]
        oof_proba[va_idx] = proba
        fold_metrics.append(compute_metrics(y_va, proba))

    elapsed = time.time() - start
    df_fm = pd.DataFrame(fold_metrics)
    summary = {
        "model": model_name,
        "roc_auc_mean": df_fm["roc_auc"].mean(),
        "roc_auc_std": df_fm["roc_auc"].std(),
        "pr_auc_mean": df_fm["pr_auc"].mean(),
        "f1_mean": df_fm["f1"].mean(),
        "precision_mean": df_fm["precision"].mean(),
        "recall_mean": df_fm["recall"].mean(),
        "accuracy_mean": df_fm["accuracy"].mean(),
        "log_loss_mean": df_fm["log_loss"].mean(),
        "fit_time_sec": round(elapsed, 2),
    }
    return summary, oof_proba

Hoàn tất tiền xử lý 5-Fold CV chống rò rỉ trong 3.28s | Số chiều sau mã hóa: 83


## 2. Đánh giá Bậc thang Mô hình Cơ sở (Baseline & Default GBDT — 5-Fold CV)
Ta huấn luyện và so sánh 6 mô hình: `Logistic Regression`, `Decision Tree`, `Random Forest`, `XGBoost`, `LightGBM`, và `CatBoost`.

In [3]:
base_models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE), True),
    ("Decision Tree", DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=RANDOM_STATE), False),
    ("Random Forest", RandomForestClassifier(n_estimators=250, max_depth=18, min_samples_leaf=4, n_jobs=-1, random_state=RANDOM_STATE), False),
    ("XGBoost (Default)", XGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE), False),
    ("LightGBM (Default)", LGBMClassifier(n_estimators=300, learning_rate=0.08, num_leaves=63, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE), False),
    ("CatBoost (Default)", CatBoostClassifier(iterations=600, learning_rate=0.08, depth=6, eval_metric="Logloss", verbose=0, thread_count=-1, random_seed=RANDOM_STATE), False),
]

cv_results = []
oof_store = {}

for name, est, scaled in base_models:
    res, oof = run_cv_model(name, est, use_scaled=scaled)
    cv_results.append(res)
    oof_store[name] = oof
    print(f"[{name:20s}] ROC-AUC: {res['roc_auc_mean']:.4f} ± {res['roc_auc_std']:.4f} | PR-AUC: {res['pr_auc_mean']:.4f} | F1: {res['f1_mean']:.4f} | Time: {res['fit_time_sec']}s")

df_base = pd.DataFrame(cv_results).sort_values("roc_auc_mean", ascending=False)
display(df_base.round(4))

[Logistic Regression ] ROC-AUC: 0.8589 ± 0.0032 | PR-AUC: 0.7143 | F1: 0.6029 | Time: 2.79s
[Decision Tree       ] ROC-AUC: 0.8870 ± 0.0011 | PR-AUC: 0.7294 | F1: 0.6698 | Time: 2.79s
[Random Forest       ] ROC-AUC: 0.9086 ± 0.0013 | PR-AUC: 0.7937 | F1: 0.6758 | Time: 9.6s
[XGBoost (Default)   ] ROC-AUC: 0.9168 ± 0.0013 | PR-AUC: 0.8096 | F1: 0.7112 | Time: 4.97s
[LightGBM (Default)  ] ROC-AUC: 0.9189 ± 0.0016 | PR-AUC: 0.8141 | F1: 0.7184 | Time: 4.22s
[CatBoost (Default)  ] ROC-AUC: 0.9165 ± 0.0029 | PR-AUC: 0.8102 | F1: 0.7119 | Time: 24.0s


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_mean,precision_mean,recall_mean,accuracy_mean,log_loss_mean,fit_time_sec
4,LightGBM (Default),0.9189,0.0016,0.8141,0.7184,0.7570,0.6839,0.8536,0.3148,4.22
3,XGBoost (Default),0.9168,0.0013,0.8096,0.7112,0.7589,0.6695,0.8516,0.3186,4.97
5,CatBoost (Default),0.9165,0.0029,0.8102,0.7119,0.7596,0.6706,0.8519,0.3198,24.00
2,Random Forest,0.9086,0.0013,0.7937,0.6758,0.7835,0.5942,0.8443,0.3462,9.60
1,Decision Tree,0.8870,0.0011,0.7294,0.6698,0.7092,0.6348,0.8292,0.3913,2.79
0,Logistic Regression,0.8589,0.0032,0.7143,0.6029,0.7155,0.5210,0.8126,0.4011,2.79


## 3. Tối ưu Siêu tham số bằng Optuna (Top 3 GBDT: LightGBM, XGBoost, CatBoost)
Sử dụng `TPESampler(seed=42)` kết hợp `MedianPruner(n_startup_trials=5, n_warmup_steps=1)` nhằm tối ưu trực tiếp **5-Fold Stratified CV ROC-AUC** cho từng mô hình trong Top 3 GBDT.

In [4]:
best_params_store = {}

# 3.1 Optuna for LightGBM
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 150),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 100),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "n_jobs": -1,
        "verbose": -1,
        "random_state": RANDOM_STATE,
    }
    aucs = []
    for step, (_, _, X_tr, y_tr, X_va, y_va) in enumerate(fold_data_unscaled):
        clf = LGBMClassifier(**params)
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, proba))
        trial.report(float(np.mean(aucs)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

t0 = time.time()
study_lgbm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study_lgbm.optimize(objective_lgbm, n_trials=25, timeout=180)
best_lgbm_params = {**study_lgbm.best_params, "subsample_freq": 1, "n_jobs": -1, "verbose": -1, "random_state": RANDOM_STATE}
best_params_store["LightGBM (Optuna)"] = best_lgbm_params
print(f"LightGBM Optuna hoàn tất ({len(study_lgbm.trials)} trials, {time.time()-t0:.1f}s) | Best CV ROC-AUC: {study_lgbm.best_value:.5f}")
print("Best LightGBM params:", study_lgbm.best_params)

res_lgbm, oof_lgbm = run_cv_model("LightGBM (Optuna)", LGBMClassifier(**best_lgbm_params), use_scaled=False)
cv_results.append(res_lgbm)
oof_store["LightGBM (Optuna)"] = oof_lgbm

LightGBM Optuna hoàn tất (16 trials, 182.0s) | Best CV ROC-AUC: 0.91978
Best LightGBM params: {'n_estimators': 910, 'learning_rate': 0.023952259500480855, 'num_leaves': 145, 'max_depth': 11, 'min_child_samples': 29, 'subsample': 0.7800129688049032, 'colsample_bytree': 0.6311991894372189, 'reg_alpha': 0.01034059327521273, 'reg_lambda': 0.9514310667418707}


In [5]:
# 3.2 Optuna for XGBoost
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1100),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "tree_method": "hist",
        "eval_metric": "logloss",
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
    }
    aucs = []
    for step, (_, _, X_tr, y_tr, X_va, y_va) in enumerate(fold_data_unscaled):
        clf = XGBClassifier(**params)
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, proba))
        trial.report(float(np.mean(aucs)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

t0 = time.time()
study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study_xgb.optimize(objective_xgb, n_trials=25, timeout=180)
best_xgb_params = {**study_xgb.best_params, "tree_method": "hist", "eval_metric": "logloss", "n_jobs": -1, "random_state": RANDOM_STATE}
best_params_store["XGBoost (Optuna)"] = best_xgb_params
print(f"XGBoost Optuna hoàn tất ({len(study_xgb.trials)} trials, {time.time()-t0:.1f}s) | Best CV ROC-AUC: {study_xgb.best_value:.5f}")
print("Best XGBoost params:", study_xgb.best_params)

res_xgb, oof_xgb = run_cv_model("XGBoost (Optuna)", XGBClassifier(**best_xgb_params), use_scaled=False)
cv_results.append(res_xgb)
oof_store["XGBoost (Optuna)"] = oof_xgb

XGBoost Optuna hoàn tất (19 trials, 182.6s) | Best CV ROC-AUC: 0.91863
Best XGBoost params: {'n_estimators': 737, 'learning_rate': 0.0326205009134931, 'max_depth': 10, 'min_child_weight': 3.857879786113614, 'subsample': 0.8316285316600499, 'colsample_bytree': 0.7717081854730012, 'gamma': 0.8291890459206518, 'reg_alpha': 0.8255226993282166, 'reg_lambda': 0.20613134431474037}


In [6]:
# 3.3 Optuna for CatBoost
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 1400),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.14, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 12.0),
        "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.5),
        "eval_metric": "Logloss",
        "verbose": 0,
        "thread_count": -1,
        "random_seed": RANDOM_STATE,
    }
    aucs = []
    for step, (_, _, X_tr, y_tr, X_va, y_va) in enumerate(fold_data_unscaled):
        clf = CatBoostClassifier(**params)
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, proba))
        trial.report(float(np.mean(aucs)), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

t0 = time.time()
study_cat = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study_cat.optimize(objective_cat, n_trials=20, timeout=200)
best_cat_params = {**study_cat.best_params, "eval_metric": "Logloss", "verbose": 0, "thread_count": -1, "random_seed": RANDOM_STATE}
best_params_store["CatBoost (Optuna)"] = best_cat_params
print(f"CatBoost Optuna hoàn tất ({len(study_cat.trials)} trials, {time.time()-t0:.1f}s) | Best CV ROC-AUC: {study_cat.best_value:.5f}")
print("Best CatBoost params:", study_cat.best_params)

res_cat, oof_cat = run_cv_model("CatBoost (Optuna)", CatBoostClassifier(**best_cat_params), use_scaled=False)
cv_results.append(res_cat)
oof_store["CatBoost (Optuna)"] = oof_cat

# Full CV Benchmark Table (9 models: 6 base + 3 tuned GBDT)
df_cv_all = pd.DataFrame(cv_results).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
print("\n=== BẢNG TỔNG HỢP KẾT QUẢ 5-FOLD STRATIFIED CV ===")
display(df_cv_all.round(4))

CatBoost Optuna hoàn tất (5 trials, 228.4s) | Best CV ROC-AUC: 0.91693
Best CatBoost params: {'iterations': 889, 'learning_rate': 0.04698459896709375, 'depth': 7, 'l2_leaf_reg': 2.53443246717246, 'random_strength': 1.2303616213380453, 'bagging_temperature': 0.5495427649405376}

=== BẢNG TỔNG HỢP KẾT QUẢ 5-FOLD STRATIFIED CV ===


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_mean,precision_mean,recall_mean,accuracy_mean,log_loss_mean,fit_time_sec
0,LightGBM (Optuna),0.9198,0.0028,0.8172,0.7187,0.7617,0.6809,0.8546,0.3141,22.83
1,LightGBM (Default),0.9189,0.0016,0.8141,0.7184,0.7570,0.6839,0.8536,0.3148,4.22
2,XGBoost (Optuna),0.9186,0.0022,0.8143,0.7161,0.7596,0.6781,0.8533,0.3160,18.72
3,CatBoost (Optuna),0.9169,0.0041,0.8111,0.7105,0.7630,0.6664,0.8522,0.3193,55.39
4,XGBoost (Default),0.9168,0.0013,0.8096,0.7112,0.7589,0.6695,0.8516,0.3186,4.97
5,CatBoost (Default),0.9165,0.0029,0.8102,0.7119,0.7596,0.6706,0.8519,0.3198,24.00
6,Random Forest,0.9086,0.0013,0.7937,0.6758,0.7835,0.5942,0.8443,0.3462,9.60
7,Decision Tree,0.8870,0.0011,0.7294,0.6698,0.7092,0.6348,0.8292,0.3913,2.79
8,Logistic Regression,0.8589,0.0032,0.7143,0.6029,0.7155,0.5210,0.8126,0.4011,2.79


## 4. Huấn luyện lại trên toàn bộ `train_strat` & Lưu các Pipeline vào `models/`
Đóng gói đầy đủ `Pipeline(preprocessor -> model)` cho các mô hình tiêu biểu và mô hình tốt nhất tổng thể, lưu định dạng `.joblib`, `.pkl`, `.csv` và `.json` để Notebook 04 đánh giá độc lập trên tập Test.

In [7]:
pipelines_to_save = {
    "baseline_logreg": Pipeline([
        ("preprocessor", build_preprocessor(scale_num=True)),
        ("model", LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
    ]),
    "random_forest": Pipeline([
        ("preprocessor", build_preprocessor(scale_num=False)),
        ("model", RandomForestClassifier(n_estimators=250, max_depth=18, min_samples_leaf=4, n_jobs=-1, random_state=RANDOM_STATE)),
    ]),
    "lgbm_tuned": Pipeline([
        ("preprocessor", build_preprocessor(scale_num=False)),
        ("model", LGBMClassifier(**best_lgbm_params)),
    ]),
    "xgb_tuned": Pipeline([
        ("preprocessor", build_preprocessor(scale_num=False)),
        ("model", XGBClassifier(**best_xgb_params)),
    ]),
    "catboost_tuned": Pipeline([
        ("preprocessor", build_preprocessor(scale_num=False)),
        ("model", CatBoostClassifier(**best_cat_params)),
    ]),
}

name_to_key = {
    "Logistic Regression": "baseline_logreg",
    "Random Forest": "random_forest",
    "LightGBM (Optuna)": "lgbm_tuned",
    "XGBoost (Optuna)": "xgb_tuned",
    "CatBoost (Optuna)": "catboost_tuned",
}

for key, pipe in pipelines_to_save.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, MODELS_DIR / f"{key}_pipeline.joblib")
    print(f"Đã huấn luyện & lưu {key}_pipeline.joblib ({time.time()-t0:.1f}s)")

# Identify overall best model (highest 5-Fold CV ROC-AUC)
best_overall_name = df_cv_all.iloc[0]["model"]
best_overall_key = name_to_key[best_overall_name]
joblib.dump(pipelines_to_save[best_overall_key], MODELS_DIR / "best_model_pipeline.joblib")

# Save feature names from fitted preprocessor
feature_names_out = list(pipelines_to_save[best_overall_key].named_steps["preprocessor"].get_feature_names_out())

# Save tables, OOF predictions, and hyperparameters
df_cv_all.to_csv(MODELS_DIR / "cv_benchmark_results.csv", index=False)

with (MODELS_DIR / "oof_predictions.pkl").open("wb") as f:
    pickle.dump({"y_train": y_train, "oof": oof_store}, f)

model_meta = {
    "best_overall_model": best_overall_name,
    "best_overall_key": best_overall_key,
    "hyperparameters": best_params_store,
    "feature_names_out": feature_names_out,
}
with (MODELS_DIR / "best_hyperparameters.json").open("w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)

print(f"\n>>> MÔ HÌNH TỐT NHẤT TỔNG THỂ (CV ROC-AUC): {best_overall_name} ({df_cv_all.iloc[0]['roc_auc_mean']:.5f})")
print("Đã lưu đầy đủ artifacts (.joblib, .pkl, .csv, .json) vào MachineLearning/models/!")

Đã huấn luyện & lưu baseline_logreg_pipeline.joblib (1.6s)
Đã huấn luyện & lưu random_forest_pipeline.joblib (3.8s)
Đã huấn luyện & lưu lgbm_tuned_pipeline.joblib (8.6s)
Đã huấn luyện & lưu xgb_tuned_pipeline.joblib (5.0s)
Đã huấn luyện & lưu catboost_tuned_pipeline.joblib (10.8s)

>>> MÔ HÌNH TỐT NHẤT TỔNG THỂ (CV ROC-AUC): LightGBM (Optuna) (0.91978)
Đã lưu đầy đủ artifacts (.joblib, .pkl, .csv, .json) vào MachineLearning/models/!
